### Databricks Structured Streaming

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import avg, count, round, col

In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.list("eventhub-scope1")

In [0]:
CONNECTION_STR = dbutils.secrets.get(
    scope="eventhub-scope1",
    key="databricksconn"
)

In [0]:
# Azure Event Hubs Connection String, Event Hub Namespace, name, and key

connectionConf = {
  'eventhubs.connectionString': sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(CONNECTION_STR),
  'eventhubs.name': 'banking-transactions'
}

In [0]:
df = spark.readStream \
    .format("eventhubs") \
    .options(**connectionConf) \
    .load()

df = df.withWatermark("enqueuedTime", "5 minutes")

df.writeStream \
    .option("checkpointLocation", "/Volumes/bankingdata/bronze/streamdatalogs") \
    .outputMode("append") \
    .format("delta") \
    .trigger(availableNow=True) \
    .toTable("bankingdata.bronze.transactions")

In [0]:
%sql
select * from bankingdata.bronze.transactions